# Extract a transect from a healpix grid

* Use interpolation over lots of points to generate a 2D (height, distance) transect from a 3D healpix grid.

In [ ]:
import cartopy.crs as ccrs
import intake
import matplotlib.pyplot as plt

import numpy as np
import healpy as hp
import xarray as xr
import easygems.healpix as egh

from utils import hp_mods, haversine

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
# Open catalog.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
# Use online if not on JASMIN.
# cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['online']

In [ ]:
# Load specific model.
sim = 'um_glm_n2560_RAL3p3_tuned_hk26'
sim_cat = cat[sim]

In [ ]:
ds3h = sim_cat(zoom=8, time='PT3H').to_dask().pipe(hp_mods)

In [ ]:
ds3h

In [ ]:
da = ds3h.clw.isel(time=1)

In [ ]:
# Define points.
start_point = [55, -40]
end_point = [45, -15]
num_points = 100

# Simple linear interpolation for coordinates
lats = np.linspace(start_point[0], end_point[0], num_points)
lons = np.linspace(start_point[1], end_point[1], num_points)

In [ ]:
# Let's see what the transect looks like on a map.
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_global()
ax.coastlines()
egh.healpix_show(da.mean(dim='pressure'))
ax.plot(lons, lats)

In [ ]:
# Convert to HEALPix internal angles (theta in [0, pi], phi in [0, 2pi])
theta = np.radians(90 - lats)
phi = np.radians(lons)

In [ ]:
# For each theta, phi, get the pix (cell ids) and weights for each pixel.
# Each has size (4, len(lon))
nside = 2**8
pix, weights = hp.get_interp_weights(nside, theta, phi, nest=True)

In [ ]:
# A transect is nothing more than an interpolation over many points, use the weights from our coords to generate this.
# transect is then the sum of the weights * the values of the field at each of pix.
# If you pass in a xr.DataArray with dims='distance', transect will have dims (pressure, distance)
transect = sum([da.isel(cell=xr.DataArray(pix[i], dims='distance')) * weights[i] for i in range(4)])

In [ ]:
# Calc distances along transect. Note, will handle non-straight transects.
distances = np.zeros(num_points)
for i in range(1, num_points):
    distances[i] = distances[i-1] + haversine(lats[i-1], lons[i-1], lats[i], lons[i])

In [ ]:
# Update transect with coord for plotting.
transect = transect.assign_coords(distance=("distance", distances, {"units": "km"}))

In [ ]:
# Now display the 2D transect. Pressure is the vertical coord so invert y-axis.
transect.plot(yincrease=False)